In [1]:
from pipeline.db import get_client
import polars as pl
import numpy as np
client = get_client()

In [56]:
df = pl.from_arrow(client.query_arrow(
     """
SELECT
  launch_speed,
  launch_angle,
  hc_x,
  hc_y,
  stand,
  game_year,
  estimated_woba_using_speedangle AS savant_xwoba,
  events
FROM pitches
WHERE launch_speed IS NOT NULL
AND launch_angle IS NOT NULL
AND events IS NOT NULL
AND estimated_ba_using_speedangle IS NOT NULL
AND hc_x IS NOT NULL
AND hc_y IS NOT NULL
"""
 )
 )
df.shape

(690170, 8)

In [58]:
df = df.with_columns(
      pl.when(pl.col("events") == "single").then(pl.lit("single"))
        .when(pl.col("events") == "double").then(pl.lit("double"))
        .when(pl.col("events") == "triple").then(pl.lit("triple"))
        .when(pl.col("events") == "home_run").then(pl.lit("hr"))
        .otherwise(pl.lit("out"))
        .alias("outcome")
  )

# Get a spray angle with respect to straight up the middle (ie theta in -90 - 90).  Weird statcast adjustment:
# https://tht.fangraphs.com/research-notebook-new-format-for-statcast-data-export-at-baseball-savant/
statcast_0 = (x0, y0) = (125.42, 198.27)
df = df.with_columns(
    [
        (pl.arctan2(pl.col("hc_x") - 125.42, 198.27 - pl.col("hc_y")) * (180 / np.pi)).alias("spray_angle_raw")
    ]
)

# Adjust for if ball is pulled (ie down the third base line is pulling for righty, but 'pushed' for a lefty
df = df.with_columns(
      pl.when(pl.col("stand") == "R")
        .then(-pl.col("spray_angle_raw"))
        .otherwise(pl.col("spray_angle_raw"))
        .alias("spray_angle_pull")
  )

In [59]:
df.head()

launch_speed,launch_angle,hc_x,hc_y,stand,game_year,savant_xwoba,events,outcome,spray_angle_raw,spray_angle_pull
f32,f32,f32,f32,str,u16,f32,str,str,f32,f32
104.199997,8.0,111.440002,100.449997,"""R""",2023,0.707,"""single""","""single""",-8.13338,8.13338
96.599998,-4.0,146.690002,158.529999,"""R""",2023,0.268,"""grounded_into_double_play""","""out""",28.156982,-28.156982
72.400002,4.0,101.519997,169.210007,"""R""",2023,0.135,"""grounded_into_double_play""","""out""",-39.435127,39.435127
105.699997,16.0,81.459999,62.740002,"""R""",2023,0.679,"""double""","""double""",-17.970793,17.970793
99.400002,-11.0,106.589996,153.949997,"""R""",2023,0.22,"""field_out""","""out""",-23.018921,23.018921


In [66]:
# Quick modeling - use pre 2025 as training
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss, classification_report

X = df.select(["launch_speed", "launch_angle"]).to_numpy()
y = df["outcome"].to_numpy()

In [61]:
train_mask = df["game_year"] < 2025
X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[~train_mask], y[~train_mask]

In [63]:
# Baseline out of the box model
model = HistGradientBoostingClassifier(loss="log_loss", random_state=42)
model.fit(X_train, y_train)
model.score(X_test, y_test) # Accuracy

0.7719644227969399

In [65]:
# Lets see how the baseline model does
probs = model.predict_proba(X_test)
print("log loss:", log_loss(y_test, probs, labels=model.classes_))
print(classification_report(y_test, model.predict(X_test)))

log loss: 0.5764298750229594
              precision    recall  f1-score   support

      double       0.42      0.16      0.23      9544
          hr       0.67      0.73      0.70      6796
         out       0.82      0.91      0.86    101946
      single       0.64      0.55      0.59     32068
      triple       0.00      0.00      0.00       754

    accuracy                           0.77    151108
   macro avg       0.51      0.47      0.48    151108
weighted avg       0.75      0.77      0.75    151108



In [67]:
# Take the calculated linear weights from fangraphs/GUTS 2025
w_2B = 1.252
w_HR = 2.037
w_1B = .882
w_3B = 1.584
w = np.array([w_2B, w_HR, 0.0, w_1B, w_3B])
my_xwoba = probs @ w

In [69]:
savant = df.filter(~train_mask)["savant_xwoba"].to_numpy()
print(f"r:    {np.corrcoef(my_xwoba, savant)[0,1]:.4f}")
print(f"MAE:  {np.mean(np.abs(my_xwoba - savant)):.4f}")

r:    0.9848
MAE:  0.0394


In [70]:
for cls in model.classes_:
      mask = (y_test == cls)
      print(f"{cls}: my_mean = {my_xwoba[mask].mean():.3f}, savant_mean = {savant[mask].mean():.3f}")

double: my_mean = 0.702, savant_mean = 0.660
hr: my_mean = 1.435, savant_mean = 1.309
out: my_mean = 0.229, savant_mean = 0.220
single: my_mean = 0.548, savant_mean = 0.553
triple: my_mean = 0.749, savant_mean = 0.691


In [74]:
triple_mask = (y_test == "triple")
triple_idx = list(model.classes_).index("triple")
print(f"P(triple) on actual triples: {probs[triple_mask,triple_idx].mean():.3f}")
print(f"P(triple) on non-triples:    {probs[~triple_mask,triple_idx].mean():.3f}")

P(triple) on actual triples: 0.017
P(triple) on non-triples:    0.006


In [76]:
# Now we can try modeling on our own features
X1 = df.select(["launch_speed", "launch_angle", "spray_angle_pull"]).to_numpy()
X1_train, y_train = X1[train_mask], y[train_mask]
X1_test,  y_test  = X1[~train_mask], y[~train_mask]

In [80]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
      "max_iter":          [200, 400, 600],
      "learning_rate":     [0.03, 0.05, 0.1, 0.2],
      "max_depth":         [3, 4, 5, 6, 12, None],
      "max_leaf_nodes":    [15, 31, 63, 127],
      "min_samples_leaf":  [20, 50, 100, 200],
      "l2_regularization": [0.0, 0.1, 1.0, 10.0],
  }

base = HistGradientBoostingClassifier(
  loss="log_loss",
  early_stopping=True,
  random_state=42,
)

search = RandomizedSearchCV(
  estimator=base,
  param_distributions=param_grid,
  n_iter=30,               # 30 random combos — increase if you
  scoring="neg_log_loss",  # the right metric
  cv=5,                    # 3-fold — 5 is slower, marginal
  n_jobs=-1,               # parallelize across cores
  verbose=0,
  random_state=42,
  refit=True,              # refits best model on full training
)

search.fit(X1_train, y_train)

print("Best log loss:", -search.best_score_)
print("Best params:  ", search.best_params_)

Best log loss: 0.454687865607179
Best params:   {'min_samples_leaf': 100, 'max_leaf_nodes': 127, 'max_iter': 200, 'max_depth': None, 'learning_rate': 0.03, 'l2_regularization': 1.0}


In [82]:
# So we can get the log loss down to ~.45.

probs = search.predict_proba(X1_test)
my_xwoba2 = probs @ w
print(f"r:    {np.corrcoef(my_xwoba2, savant)[0,1]:.4f}")
print(f"MAE:  {np.mean(np.abs(my_xwoba2 - savant)):.4f}")

r:    0.8740
MAE:  0.1342


In [ ]:
# This actually decreases correlation with statcast and increases error.  Not necessarily a bad thing, just likely a change in how things are calculated